In [1]:
from __future__ import (absolute_import, division, print_function,
                        unicode_literals)

import datetime  # For datetime objects
import os.path  # To manage paths
import sys  # To find out the script name (in argv[0])

# Import the backtrader platform
import backtrader as bt

In [2]:
# Create a Stratey
class TestStrategy(bt.Strategy):
    params = (
        ('maperiod', 15),
    )

    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), txt))

    def __init__(self):
        # Keep a reference to the "close" line in the data[0] dataseries
        self.dataclose = self.datas[0].close

        # To keep track of pending orders and buy price/commission
        self.order = None
        self.buyprice = None
        self.buycomm = None

        self.prenext_counter = 0

        # Add a MovingAverageSimple indicator
        self.sma = bt.indicators.SimpleMovingAverage(
            self.datas[0], period=self.params.maperiod)

    def notify_order(self, order):
        self.log('NotifyOrder Status:{}'.format(order.Status[order.status]))
        if order.status in [order.Submitted, order.Accepted]:
            # Buy/Sell order submitted/accepted to/by broker - Nothing to do
            return

        # Check if an order has been completed
        # Attention: broker could reject order if not enough cash
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(
                    'BUY EXECUTED, Price: %.3f, Cost: %.3f, Comm %.2f' %
                    (order.executed.price,
                     order.executed.value,
                     order.executed.comm))

                self.buyprice = order.executed.price
                self.buycomm = order.executed.comm
            else:  # Sell
                self.log('SELL EXECUTED, Price: %.3f, Cost: %.3f, Comm %.2f' %
                         (order.executed.price,
                          order.executed.value,
                          order.executed.comm))

            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log('Order Canceled/Margin/Rejected')

        # Write down: no pending order
        self.order = None

    def notify_trade(self, trade):
        self.log('NotifyTrade Status:{}'.format(trade.status_names[trade.status]))
        if not trade.isclosed:
            return

        self.log('OPERATION PROFIT, GROSS %.2f, NET %.2f' %
                 (trade.pnl, trade.pnlcomm))

    def next(self):
        # Simply log the closing price of the series from the reference
        self.log('Next Open:{} Close:{}'.format(self.datas[0].open[0], self.dataclose[0]))

        # Check if an order is pending ... if yes, we cannot send a 2nd one
        if self.order:
            return

        # Check if we are in the market
        if not self.position:

            # Not yet ... we MIGHT BUY if ...
            if self.dataclose[0] > self.sma[0]:

                # BUY, BUY, BUY!!! (with all possible default parameters)
                self.log('BUY CREATE, %.3f' % self.dataclose[0])

                # Keep track of the created order to avoid a 2nd order
                self.order = self.buy()

        else:

            if self.dataclose[0] < self.sma[0]:
                # SELL, SELL, SELL!!! (with all possible default parameters)
                self.log('SELL CREATE, %.3f' % self.dataclose[0])

                # Keep track of the created order to avoid a 2nd order
                self.order = self.sell()

    def nextstart(self):
        self.log('NextStart Called')

    def prenext(self):
        self.prenext_counter = self.prenext_counter+1
        self.log('PreNext Called C:{}'.format(self.prenext_counter))

    def start(self):
        self.log('Start Called')

    def stop(self):
        self.log('Stop Called')

    def notify_cashvalue(self, cash, value):
        self.log('NotifyCashValue C:{} V:{}'.format(cash, value))

    def notify_fund(self, cash, value, fundvalue, shares):
        self.log('NotifyFund C:{} V:{} F:{} S:{}'.format(cash, value, fundvalue, shares))



In [3]:
# Create a cerebro entity
cerebro = bt.Cerebro()

# Add a strategy
cerebro.addstrategy(TestStrategy)

# Create a Data Feed
data = bt.feeds.BacktraderCSVData(dataname='./510300_btf.csv')

# Add the Data Feed to Cerebro
cerebro.adddata(data)

# Set our desired cash start
cerebro.broker.setcash(10000.0)

# Add a FixedSize sizer according to the stake
cerebro.addsizer(bt.sizers.FixedSize, stake=10)

# Set the commission
cerebro.broker.setcommission(commission=0.0)

# Print out the starting conditions
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())

# Run over everything
cerebro.run()

# Print out the final result
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

Starting Portfolio Value: 10000.00
2019-12-31, Start Called
2018-01-02, NotifyCashValue C:10000.0 V:10000.0
2018-01-02, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-01-02, PreNext Called C:1
2018-01-03, NotifyCashValue C:10000.0 V:10000.0
2018-01-03, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-01-03, PreNext Called C:2
2018-01-04, NotifyCashValue C:10000.0 V:10000.0
2018-01-04, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-01-04, PreNext Called C:3
2018-01-05, NotifyCashValue C:10000.0 V:10000.0
2018-01-05, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-01-05, PreNext Called C:4
2018-01-08, NotifyCashValue C:10000.0 V:10000.0
2018-01-08, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-01-08, PreNext Called C:5
2018-01-09, NotifyCashValue C:10000.0 V:10000.0
2018-01-09, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-01-09, PreNext Called C:6
2018-01-10, NotifyCashValue C:10000.0 V:10000.0
2018-01-10, NotifyFund C:10000.0 V:10000.0 F:100.0 S:100.0
2018-

In [4]:
# Plot the result
cerebro.plot()

<IPython.core.display.Javascript object>

[[<Figure size 640x480 with 4 Axes>]]

In [5]:
cerebro.plot?

Signature:
cerebro.plot(
    plotter=None,
    numfigs=1,
    iplot=True,
    start=None,
    end=None,
    width=16,
    height=9,
    dpi=300,
    tight=True,
    use=None,
    **kwargs,
)
Docstring:
Plots the strategies inside cerebro

If ``plotter`` is None a default ``Plot`` instance is created and
``kwargs`` are passed to it during instantiation.

``numfigs`` split the plot in the indicated number of charts reducing
chart density if wished

``iplot``: if ``True`` and running in a ``notebook`` the charts will be
displayed inline

``use``: set it to the name of the desired matplotlib backend. It will
take precedence over ``iplot``

``start``: An index to the datetime line array of the strategy or a
``datetime.date``, ``datetime.datetime`` instance indicating the start
of the plot

``end``: An index to the datetime line array of the strategy or a
``datetime.date``, ``datetime.datetime`` instance indicating the end
of the plot

``width``: in inches of the saved figure

``height``: in